# 03 · CNN 입문 — MNIST로 맛보기

**왜 MNIST?** 병리 이미지보다 가볍고 (28×28 흑백), CNN의 핵심 패턴을 빠르게 체득 가능.
이 노트북이 끝나면 `03-pathology/`의 의료 영상도 같은 틀로 다룰 수 있다.

**CNN의 핵심 아이디어**:
- **국소성 (locality)**: 픽셀은 근처 픽셀과 가장 관련이 깊다 → 작은 필터만 써도 됨.
- **가중치 공유 (weight sharing)**: 같은 필터를 이미지 전체에 sliding → 파라미터 수 대폭 감소.
- **계층적 특징**: 앞 층은 엣지/선, 뒤 층은 텍스처/패턴/객체.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device = {device}')

## 1. 데이터 — MNIST

CPU만 써도 빠르게 돌도록 **서브셋**만 사용 (train 6000, test 1000).
전부 쓰고 싶으면 `Subset` 라인을 주석 처리.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),           # PIL → Tensor (0~1로 정규화도 같이)
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 평균/표준편차
])

import os
os.makedirs('./data', exist_ok=True)
full_train = datasets.MNIST('./data', train=True,  download=True, transform=transform)
full_test  = datasets.MNIST('./data', train=False, download=True, transform=transform)

# 서브셋 (빠른 실험용). 전체 쓰려면 아래 두 줄 주석 처리.
train_ds = Subset(full_train, range(6000))
test_ds  = Subset(full_test,  range(1000))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

print(f'train {len(train_ds)}, test {len(test_ds)}')

In [ ]:
# 샘플 보기
import matplotlib.pyplot as plt
xb, yb = next(iter(train_loader))
print(f'배치 shape: {xb.shape} (B, C, H, W)')
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(xb[i, 0], cmap='gray'); ax.set_title(int(yb[i])); ax.axis('off')
plt.show()

## 2. CNN 모델

```
Input (1, 28, 28)
  → Conv2d(1 → 16) + ReLU + MaxPool → (16, 14, 14)
  → Conv2d(16 → 32) + ReLU + MaxPool → (32, 7, 7)
  → Flatten → Linear(32·7·7 → 64) → ReLU → Dropout → Linear(64 → 10)
```

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, n_classes)
        self.drop  = nn.Dropout(0.25)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # (16, 14, 14)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # (32, 7, 7)
        x = x.flatten(1)                             # (B, 32*7*7)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)

model = SmallCNN().to(device)
print(model)
print(f'파라미터 수: {sum(p.numel() for p in model.parameters()):,}')

## 3. 학습

In [ ]:
loss_fn = nn.CrossEntropyLoss()            # 다중 분류 표준
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def accuracy(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(1)
            correct += (pred == yb).sum().item()
            total   += len(yb)
    return correct / total

EPOCHS = 5
history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(yb); n += len(yb)
    train_loss = running / n
    test_acc = accuracy(model, test_loader)
    history.append((train_loss, test_acc))
    print(f'epoch {epoch} | train_loss {train_loss:.4f} | test_acc {test_acc:.4f}')

## 4. 필터 시각화 — "CNN은 뭘 보고 있나?"

첫 번째 conv layer의 필터를 이미지로 본다. 학습된 후엔 엣지/패턴 검출기처럼 보일 것.

In [ ]:
filters = model.conv1.weight.data.cpu().numpy()   # (16, 1, 3, 3)
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i, 0], cmap='gray')
    ax.axis('off')
plt.suptitle('Conv1 학습된 3×3 필터들')
plt.show()

In [ ]:
# 첫 conv의 feature map도 본다 — 입력 이미지가 어떻게 변하는지
sample, label = train_ds[0]
with torch.no_grad():
    fmaps = F.relu(model.conv1(sample.unsqueeze(0).to(device))).cpu().numpy()[0]
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(fmaps[i], cmap='viridis'); ax.axis('off')
plt.suptitle(f'입력 {label} → conv1 feature maps (16장)')
plt.show()

## 5. Layer 3으로 가는 다리

지금 만든 `SmallCNN`을 조금만 바꾸면 **병리 이미지 분류**가 된다:
- 입력 채널: 1 → 3 (RGB)
- 입력 크기: 28×28 → 96×96 (PatchCamelyon)
- 출력 클래스: 10 → 2 (전이 있음/없음)
- 더 깊은 구조 (ResNet18 transfer learning으로 바로 점프 가능)

### AI agent에 물어볼 것
1. "Conv2d의 padding, stride가 output 크기에 미치는 공식 설명해줘"
2. "Max pooling을 뺀 CNN vs 넣은 CNN, 파라미터 수와 성능이 어떻게 달라질까?"
3. "Transfer learning이 왜 의료 영상에서 특히 중요해? (데이터가 왜 적은지 포함)"

### Layer 2 끝 — 정리 체크리스트
- [ ] Tensor, autograd, 경사하강 직접 돌림
- [ ] `nn.Module`로 MLP / CNN 정의 가능
- [ ] Dataset / DataLoader 패턴 익힘
- [ ] 학습 루프 템플릿이 머릿속에 있음
- [ ] train/eval mode 구분 이유 앎
- [ ] CNN의 conv → pool → fc 구조 이해

→ `03-pathology/`로 이동 (UNLV Project 1 미니)